In [1]:
import os
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Sequential
import torch.optim as optim
import voxelmorph as vxm
import neurite as ne
import scipy.ndimage

os.environ['VXM_BACKEND'] = 'pytorch'

backend:pytorch
Pytorch


In [2]:
os.environ.get('VXM_BACKEND')

'pytorch'

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [4]:
# 画像を読み込み
x_train = np.load('Data/TrainData_NoBed.npz')['Train']
x_train = np.transpose(x_train, (3, 0, 1, 2))

print('Resized train vol_shape:', x_train.shape[1:])
print('Resized train shape:', x_train.shape)

Resized train vol_shape: (128, 256, 256)
Resized train shape: (400, 128, 256, 256)


In [5]:
import torch

def vxm_data_generator(x_data, batch_size):
    vol_shape = x_data.shape[1:]  # データ形状を取得
    ndims = len(vol_shape)
    
    zero_phi = np.zeros([batch_size, *vol_shape, ndims])
    
    while True:
        idx1 = np.random.randint(0, x_data.shape[0], size=batch_size)
        moving_images = x_data[idx1, ..., np.newaxis]
        # ファインチューニングでは同じ症例同士のペアを避ける
        idx2 = np.random.randint(0, x_data.shape[0], size=batch_size)
        while np.any(idx2 == idx1):
            same_case = idx2 == idx1
            idx2[same_case] = np.random.randint(0, x_data.shape[0], size=same_case.sum())
        fixed_images = x_data[idx2, ..., np.newaxis]

        # TensorFlowからPyTorchのデータ形式に変換
        moving_images = torch.tensor(moving_images).permute(0, 4, 1, 2, 3).float()
        fixed_images = torch.tensor(fixed_images).permute(0, 4, 1, 2, 3).float()

        # チャンネルを最初の次元に追加
        moving_images = moving_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動
        fixed_images = fixed_images.permute(0, 1, 2, 3, 4)  # チャンネルを最初の次元に移動

        inputs = [moving_images, fixed_images]
        outputs = [fixed_images, zero_phi]

        yield (inputs, outputs)

In [6]:
train_generator = vxm_data_generator(x_train, batch_size=2)
in_sample, out_sample = next(train_generator)

# in_sampleとout_sampleの内容を確認する
print("Input Sample Shapes:")
print("Moving Images Shape:", in_sample[0].shape)
print("Fixed Images Shape:", in_sample[1].shape)

print("\nOutput Sample Shapes:")
print("Moved Images (Fixed) Shape:", out_sample[0].shape)
print("Zero Gradient Shape:", out_sample[1].shape)

Input Sample Shapes:
Moving Images Shape: torch.Size([2, 1, 128, 256, 256])
Fixed Images Shape: torch.Size([2, 1, 128, 256, 256])

Output Sample Shapes:
Moved Images (Fixed) Shape: torch.Size([2, 1, 128, 256, 256])
Zero Gradient Shape: (2, 128, 256, 256, 3)


In [7]:
mse_loss = vxm.losses.MSE().loss
grad_loss = vxm.losses.Grad('l2').loss

def total_loss(y_true, y_pred):
    mse = mse_loss(y_true, y_pred)
    grad = grad_loss(y_true, y_pred)
    return mse + 0.01 * grad, mse, grad
#     return mse_loss(y_true, y_pred)

def MSE_Loss(y_true, y_pred):
    y_true = y_true.to(device)
    y_pred = y_pred.to(device)
    mse = mse_loss(y_true, y_pred)
    return mse

def lncc_loss(I, J, window=9, eps=1e-5):
    # I, J: (B, 1, D, H, W)
    padding = window // 2
    weight = torch.ones(1, 1, window, window, window, device=I.device)

    I2 = I * I
    J2 = J * J
    IJ = I * J

    I_sum = F.conv3d(I, weight, padding=padding)
    J_sum = F.conv3d(J, weight, padding=padding)
    I2_sum = F.conv3d(I2, weight, padding=padding)
    J2_sum = F.conv3d(J2, weight, padding=padding)
    IJ_sum = F.conv3d(IJ, weight, padding=padding)

    win_size = window ** 3
    u_I = I_sum / win_size
    u_J = J_sum / win_size

    cross = IJ_sum - u_J * I_sum - u_I * J_sum + u_I * u_J * win_size
    I_var = I2_sum - 2 * u_I * I_sum + u_I * u_I * win_size
    J_var = J2_sum - 2 * u_J * J_sum + u_J * u_J * win_size

    lncc = cross * cross / (I_var * J_var + eps)
    return -torch.mean(lncc)  # maximize LNCC → minimize -LNCC

In [8]:
# configure unet input shape (concatenation of moving and fixed images)
ndim = 3
unet_input_features = 2
# inshape = (*x_train.shape[1:], unet_input_features)

nb_features = [
    [32, 64, 64, 64, 64],
    [64, 64, 64, 64, 64, 32, 16, 16]
]


In [9]:
import voxelmorph as vxm
import inspect

print(vxm.__file__)
print(vxm.networks.__file__)
print([name for name in dir(vxm.networks) if "VxmDense" in name])

C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\__init__.py
C:\Users\user\anaconda3\envs\nn\lib\site-packages\voxelmorph\torch\networks.py
['VxmDense', 'VxmDense1', 'VxmDense2', 'VxmDense_128_256', 'VxmDense_128_256_256']


In [10]:
model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0)
model3D.to(device)
optimizer = optim.Adam(model3D.parameters(), lr=1e-4)

transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)
transformer256 = vxm.layers.SpatialTransformer((128, 256, 256)).to(device)

[64, 128, 128]


C:\Users\user\anaconda3\envs\nn\lib\site-packages\torch\functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [38]:
import math
from pathlib import Path
import torch
import matplotlib.pyplot as plt

band_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

wavelet_vis_enabled = False
wavelet_vis_every = 100
wavelet_vis_dir = Path('wavelet_stage_outputs')
wavelet_vis_dir.mkdir(exist_ok=True)

class Haar3DAnalysisOnly(nn.Module):
    def __init__(self):
        super().__init__()

        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2.0)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2.0)

        filters = []
        names = []

        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = names

    def forward(self, x):
        x = F.pad(x, (0, 1, 0, 1, 0, 1))
        return F.conv3d(x, self.weight, stride=1, padding=0)

def analysis_filter_3d(x, analysis_layer):
    return analysis_layer(x)

def down_sampling_3d(w):
    return w[:, :, ::2, ::2, ::2]

def up_sampling_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(
        B, C, D * 2, H * 2, W * 2,
        dtype=w_down.dtype,
        device=w_down.device
    )
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

def create_synthesis_filters(device):
    low = torch.tensor([1.0, 1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)
    high = torch.tensor([1.0, -1.0], dtype=torch.float32, device=device) / math.sqrt(2.0)

    filters = torch.stack([
        make_3d_filter(low, low, low),
        make_3d_filter(low, low, high),
        make_3d_filter(low, high, low),
        make_3d_filter(low, high, high),
        make_3d_filter(high, low, low),
        make_3d_filter(high, low, high),
        make_3d_filter(high, high, low),
        make_3d_filter(high, high, high),
    ], dim=0)

    filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
    return filters

def synthesis_filter_3d(w_up, synthesis_filters):
    B, C, D, H, W = w_up.shape
    filtered_bands = []

    for i in range(C):
        band = w_up[:, i:i + 1, :, :, :]
        kernel = synthesis_filters[i:i + 1]
        filtered = F.conv3d(band, kernel, stride=1, padding=1)
        filtered = filtered[:, :, :D, :H, :W]
        filtered_bands.append(filtered)

    filtered_bands = torch.cat(filtered_bands, dim=1)
    reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
    return reconstructed, filtered_bands

analysis = Haar3DAnalysisOnly().to(device)
synthesis_filters = create_synthesis_filters(device)
analysis_names = analysis.names

In [ ]:
from pathlib import Path
import csv
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ==========================================
# カリキュラム学習・途中経過観察の設定
# ==========================================
OBSERVE_EVERY = 2000
OBSERVE_PATIENT_ID = 0
# 正負両方向を同じ絶対値で評価する
OBSERVE_SHIFTS = [-40, -30, -20, -10, 0, 10, 20, 30, 40]
OBSERVE_RANDOM_SEED = 20260803
OBSERVE_RANDOM_MAX_SHIFT = 40.0

OBSERVE_DIR = Path("curriculum_observation_inverse_consistency")
OBSERVE_DIR.mkdir(exist_ok=True)

OBSERVE_CSV = OBSERVE_DIR / "curriculum_observation_metrics.csv"

# 観察用の患者画像を固定
observation_moving = torch.from_numpy(
    x_train[OBSERVE_PATIENT_ID:OBSERVE_PATIENT_ID + 1]
).unsqueeze(1).to(
    device=device,
    dtype=torch.float32
)


def calculate_rmse(reference, prediction):
    return torch.sqrt(
        torch.mean((reference - prediction) ** 2)
    ).item()


def calculate_dvf_rmse(reference_flow, predicted_flow):
    return torch.sqrt(
        torch.mean((reference_flow - predicted_flow) ** 2)
    ).item()


def observe_curriculum_progress(
    model,
    epoch,
    moving_image,
    shifts,
):
    """
    固定患者・固定x方向変位を使い、
    カリキュラム学習途中の性能を表示・保存する。
    """

    was_training = model.training
    model.eval()

    epoch_dir = OBSERVE_DIR / f"epoch_{epoch:06d}"
    epoch_dir.mkdir(exist_ok=True)

    result_rows = []

    with torch.no_grad():
        moving_analysis = analysis_filter_3d(
            moving_image,
            analysis
        )

        moving_w = down_sampling_3d(
            moving_analysis
        ).to(device)

        figure, axes = plt.subplots(
            len(shifts),
            4,
            figsize=(16, 4 * len(shifts)),
            constrained_layout=True
        )

        if len(shifts) == 1:
            axes = np.expand_dims(axes, axis=0)

        observation_slice = moving_image.shape[2] // 2

        moving_np = moving_image[
            0, 0, observation_slice
        ].detach().cpu().numpy()

        display_vmin, display_vmax = np.percentile(
            moving_np,
            [1, 99]
        )

        for row_index, shift_pixels in enumerate(shifts):

            # フル解像度の正解DVF
            # channel 0=z, 1=y, 2=x
            true_flow_full = torch.zeros(
                (
                    1,
                    3,
                    moving_image.shape[2],
                    moving_image.shape[3],
                    moving_image.shape[4]
                ),
                dtype=moving_image.dtype,
                device=device
            )

            true_flow_full[:, 2] = float(shift_pixels)

            # 正解画像 Moving′
            moving_prime = transformer256(
                moving_image,
                true_flow_full
            )

            moving_prime_analysis = analysis_filter_3d(
                moving_prime,
                analysis
            )

            moving_prime_w = down_sampling_3d(
                moving_prime_analysis
            ).to(device)

            # DVF予測
            predicted_flow = model(
                moving_w,
                moving_prime_w
            )

            # 8バンドへ同じDVFを適用
            warped_bands = [
                transformer(
                    moving_w[:, band:band + 1],
                    predicted_flow
                )
                for band in range(moving_w.shape[1])
            ]

            moved_w = torch.cat(
                warped_bands,
                dim=1
            )

            moved_up = up_sampling_3d(
                moved_w
            )

            moved, _ = synthesis_filter_3d(
                moved_up,
                synthesis_filters
            )

            moved = moved.to(device)

            # 低解像度DVFの正解
            true_flow_low = torch.zeros_like(
                predicted_flow
            )

            true_flow_low[:, 2] = (
                float(shift_pixels) / 2.0
            )

            image_rmse_before = calculate_rmse(
                moving_prime,
                moving_image
            )

            image_rmse_after = calculate_rmse(
                moving_prime,
                moved
            )

            dvf_rmse = calculate_dvf_rmse(
                true_flow_low,
                predicted_flow
            )

            predicted_x_mean = (
                predicted_flow[:, 2]
                .mean()
                .item()
            )

            result_rows.append({
                "epoch": epoch,
                "shift_pixels_fullres_x": shift_pixels,
                "true_flow_x_lowres": shift_pixels / 2.0,
                "predicted_flow_x_mean_lowres": predicted_x_mean,
                "rmse_before": image_rmse_before,
                "rmse_after": image_rmse_after,
                "dvf_rmse": dvf_rmse,
            })

            moving_prime_np = moving_prime[
                0, 0, observation_slice
            ].detach().cpu().numpy()

            moved_np = moved[
                0, 0, observation_slice
            ].detach().cpu().numpy()

            predicted_flow_x_np = predicted_flow[
                0,
                2,
                predicted_flow.shape[2] // 2
            ].detach().cpu().numpy()

            axes[row_index, 0].imshow(
                moving_np,
                cmap="gray",
                vmin=display_vmin,
                vmax=display_vmax
            )
            axes[row_index, 0].set_title(
                f"Moving\nEpoch {epoch}"
            )

            axes[row_index, 1].imshow(
                moving_prime_np,
                cmap="gray",
                vmin=display_vmin,
                vmax=display_vmax
            )
            axes[row_index, 1].set_title(
                f"Moving'\nx shift={shift_pixels}px"
            )

            axes[row_index, 2].imshow(
                moved_np,
                cmap="gray",
                vmin=display_vmin,
                vmax=display_vmax
            )
            axes[row_index, 2].set_title(
                f"Moved\nRMSE={image_rmse_after:.5f}"
            )

            dvf_image = axes[row_index, 3].imshow(
                predicted_flow_x_np,
                cmap="jet"
            )

            axes[row_index, 3].set_title(
                "Predicted DVF x\n"
                f"mean={predicted_x_mean:.3f}, "
                f"RMSE={dvf_rmse:.3f}"
            )

            figure.colorbar(
                dvf_image,
                ax=axes[row_index, 3],
                fraction=0.046,
                pad=0.04
            )

            for column_index in range(4):
                axes[row_index, column_index].axis("off")

        figure.suptitle(
            f"Curriculum learning observation — Epoch {epoch}",
            fontsize=16
        )

        figure_path = (
            epoch_dir
            / f"curriculum_observation_epoch_{epoch:06d}.png"
        )

        figure.savefig(
            figure_path,
            dpi=180,
            bbox_inches="tight"
        )

        plt.show()
        plt.close(figure)

    csv_exists = OBSERVE_CSV.exists()

    with OBSERVE_CSV.open(
        "a",
        newline="",
        encoding="utf-8"
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=result_rows[0].keys()
        )

        if not csv_exists:
            writer.writeheader()

        writer.writerows(result_rows)

    if was_training:
        model.train()

    print(
        f"[Observation] Epoch {epoch}: "
        f"saved to {epoch_dir}"
    )


def observe_random_deformation_progress(model, epoch, moving_image):
    """学習と同型の固定ランダム・滑らかDVFで性能を観察する。"""
    was_training = model.training
    model.eval()
    epoch_dir = OBSERVE_DIR / f"epoch_{epoch:06d}"
    epoch_dir.mkdir(exist_ok=True)

    # 毎回まったく同じDVFを作るため、局所Generatorを使う。
    random_generator = torch.Generator(device="cpu")
    random_generator.manual_seed(OBSERVE_RANDOM_SEED)
    true_flow_full = (
        torch.rand((1, 3, 8, 16, 16), generator=random_generator) * 2 - 1
    ) * OBSERVE_RANDOM_MAX_SHIFT
    true_flow_full = gaussian_smooth_3d(true_flow_full, sigma=2.0)
    true_flow_full = F.interpolate(
        true_flow_full, size=(128, 256, 256), mode="trilinear", align_corners=False
    ).to(device)

    with torch.no_grad():
        moving_prime = transformer256(moving_image, true_flow_full)
        moving_w = down_sampling_3d(analysis_filter_3d(moving_image, analysis)).to(device)
        moving_prime_w = down_sampling_3d(
            analysis_filter_3d(moving_prime, analysis)
        ).to(device)
        predicted_flow = model(moving_w, moving_prime_w)
        warped_bands = [
            transformer(moving_w[:, band:band + 1], predicted_flow)
            for band in range(moving_w.shape[1])
        ]
        moved, _ = synthesis_filter_3d(
            up_sampling_3d(torch.cat(warped_bands, dim=1)), synthesis_filters
        )

        true_flow_low = F.interpolate(
            true_flow_full, size=predicted_flow.shape[2:], mode="trilinear", align_corners=False
        )
        rmse_before = calculate_rmse(moving_prime, moving_image)
        rmse_after = calculate_rmse(moving_prime, moved)
        dvf_rmse = calculate_dvf_rmse(true_flow_low, predicted_flow)

        result_row = {
            "epoch": epoch, "seed": OBSERVE_RANDOM_SEED,
            "max_initial_shift": OBSERVE_RANDOM_MAX_SHIFT,
            "rmse_before": rmse_before, "rmse_after": rmse_after,
            "dvf_rmse": dvf_rmse,
            "true_flow_absmax_lowres": true_flow_low.abs().max().item(),
            "predicted_flow_absmax_lowres": predicted_flow.abs().max().item(),
        }

        slice_index = moving_image.shape[2] // 2
        flow_slice = predicted_flow.shape[2] // 2
        figure, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
        image_vmin, image_vmax = np.percentile(
            moving_image[0, 0, slice_index].detach().cpu().numpy(), [1, 99]
        )
        for ax, image, title in [
            (axes[0, 0], moving_image, "Moving"),
            (axes[0, 1], moving_prime, "Target (random DVF)"),
            (axes[0, 2], moved, f"Moved (RMSE={rmse_after:.5f})"),
        ]:
            ax.imshow(image[0, 0, slice_index].detach().cpu(), cmap="gray", vmin=image_vmin, vmax=image_vmax)
            ax.set_title(title)
            ax.axis("off")
        for component, name in enumerate(["z", "y", "x"]):
            error = (predicted_flow[0, component, flow_slice] - true_flow_low[0, component, flow_slice]).detach().cpu()
            dvf = axes[1, component].imshow(error, cmap="coolwarm")
            axes[1, component].set_title(f"{name}-DVF error")
            axes[1, component].axis("off")
            figure.colorbar(dvf, ax=axes[1, component], fraction=0.046, pad=0.04)
        figure.suptitle(f"Fixed random deformation observation — Epoch {epoch}; DVF RMSE={dvf_rmse:.4f}")
        figure.savefig(epoch_dir / f"random_deformation_epoch_{epoch:06d}.png", dpi=180, bbox_inches="tight")
        plt.show()
        plt.close(figure)

    random_csv = OBSERVE_DIR / "random_deformation_observation_metrics.csv"
    csv_exists = random_csv.exists()
    with random_csv.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=result_row.keys())
        if not csv_exists:
            writer.writeheader()
        writer.writerow(result_row)
    if was_training:
        model.train()
    print(f"[Random deformation observation] Epoch {epoch}: saved to {epoch_dir}")

In [ ]:
# ==========================================
# 逆写像整合性制約（DWT後の低解像度DVFで計算）
# ==========================================
INVERSE_CONSISTENCY_WEIGHT = 1.0

def compose_dvfs(first_flow, second_flow):
    """first_flow の後に second_flow を適用した合成DVF。

    VoxelMorphのflowはボクセル単位である。grid_sample用の正規化座標を
    手作業で作らず、既存のSpatialTransformerでflowをサンプルする。
    """
    return first_flow + transformer(second_flow, first_flow)


def inverse_consistency_loss(flow_ab, flow_ba):
    # A→B→A と B→A→B の両方が恒等変換になるよう制約する。
    cycle_on_b = compose_dvfs(flow_ab, flow_ba)
    cycle_on_a = compose_dvfs(flow_ba, flow_ab)
    return 0.5 * (torch.mean(cycle_on_b ** 2) + torch.mean(cycle_on_a ** 2))


In [ ]:
# DWT + inverse-consistency pretraining
from tqdm.notebook import tqdm
from IPython.display import clear_output
import matplotlib.pyplot as plt

def gaussian_smooth_3d(tensor, kernel_size=5, sigma=1.0):
    """学習時と評価時で共通の3D Gaussian smoothing。"""
    from scipy.ndimage import gaussian_filter
    smoothed_np = gaussian_filter(tensor.detach().cpu().numpy(), sigma=[0, 0, sigma, sigma, sigma])
    return torch.tensor(smoothed_np, dtype=torch.float32, device=tensor.device)

epochs = 80000
shift_range = 1

losses = []
loss_vecs_ab = []
loss_images_ab = []
loss_images_ba = []
loss_inverses = []

for epoch in tqdm(range(epochs), desc='DWT + inverse-consistency pretraining'):
    if epoch % 2000 == 0 and epoch > 0:
        shift_range += 1
        print(f"Epoch {epoch}: Increasing shift range to ±{shift_range} pixels.")

    train_batch, _ = next(train_generator)
    moving_images = torch.as_tensor(train_batch[0], dtype=torch.float32, device=device)

    # 学習用の正解は滑らかなランダム3D DVF（z,y,xの全方向）。
    displacement_field = (torch.rand((2, 3, 8, 16, 16), dtype=torch.float32) * 2 - 1) * shift_range
    displacement_field = gaussian_smooth_3d(displacement_field.to(device), sigma=2.0)
    displacement_field = F.interpolate(
        displacement_field, size=(128, 256, 256), mode='trilinear', align_corners=False
    )
    displacement_field128 = F.interpolate(
        displacement_field, size=(64, 128, 128), mode='trilinear', align_corners=False
    )
    moving_prime = transformer256(moving_images, displacement_field)

    moving_w = down_sampling_3d(analysis_filter_3d(moving_images, analysis)).to(device)
    moving_prime_w = down_sampling_3d(analysis_filter_3d(moving_prime, analysis)).to(device)

    optimizer.zero_grad()

    # 順方向 A→B
    flow_ab = model3D(moving_w, moving_prime_w)
    moved_ab, _ = synthesis_filter_3d(
        up_sampling_3d(transformer(moving_w, flow_ab)), synthesis_filters
    )

    # 逆方向 B→A。任意DVFの厳密な逆は -DVF ではないため、DVF教師損失は置かない。
    flow_ba = model3D(moving_prime_w, moving_w)
    moved_ba, _ = synthesis_filter_3d(
        up_sampling_3d(transformer(moving_prime_w, flow_ba)), synthesis_filters
    )

    loss_vec_ab = MSE_Loss(displacement_field128, flow_ab) * 0.01
    loss_image_ab = MSE_Loss(moving_prime, moved_ab) * 100.0
    loss_image_ba = MSE_Loss(moving_images, moved_ba) * 100.0
    loss_inverse = inverse_consistency_loss(flow_ab, flow_ba)
    loss = loss_vec_ab + loss_image_ab + loss_image_ba + INVERSE_CONSISTENCY_WEIGHT * loss_inverse

    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    loss_vecs_ab.append(loss_vec_ab.item())
    loss_images_ab.append(loss_image_ab.item())
    loss_images_ba.append(loss_image_ba.item())
    loss_inverses.append(loss_inverse.item())

    # 固定患者・正負一定シフト・固定ランダム3D DVFで、目視と数値の両方を保存。
    if epoch == 0 or (epoch + 1) % OBSERVE_EVERY == 0 or (epoch + 1) == epochs:
        observe_curriculum_progress(model3D, epoch + 1, observation_moving, OBSERVE_SHIFTS)
        observe_random_deformation_progress(model3D, epoch + 1, observation_moving)

    if (epoch + 1) % 100 == 0:
        torch.save(model3D.state_dict(), 'model_analysis_pipeline_pretrain_inverse_consistency.pth')

    if (epoch + 1) % OBSERVE_EVERY == 0:
        checkpoint_dir = Path('curriculum_inverse_consistency_checkpoints')
        checkpoint_dir.mkdir(exist_ok=True)
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model3D.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'shift_range': shift_range,
            'inverse_consistency_weight': INVERSE_CONSISTENCY_WEIGHT,
            'loss_inverse': loss_inverse.item(),
        }, checkpoint_dir / f'pretrain_inverse_epoch_{epoch + 1:05d}.pth')

    if epoch % 10 == 0:
        clear_output(wait=True)
        plt.figure(figsize=(11, 5))
        plt.plot(losses, label='total')
        plt.plot(loss_vecs_ab, label='forward DVF')
        plt.plot(loss_images_ab, label='A→B image')
        plt.plot(loss_images_ba, label='B→A image')
        plt.plot(loss_inverses, label='inverse consistency')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('DWT + inverse-consistency training loss')
        plt.legend()
        plt.grid(True)
        plt.show()

    print(
        f'Epoch {epoch + 1}/{epochs}, total={loss:.5f}, '
        f'forward_dvf={loss_vec_ab:.5f}, A→B={loss_image_ab:.5f}, '
        f'B→A={loss_image_ba:.5f}, inverse={loss_inverse:.5f}, ±{shift_range}'
    )


In [ ]:
# DWT + inverse-consistency fine-tuning
from tqdm.notebook import tqdm

pretrained_path = 'model_analysis_pipeline_pretrain_inverse_consistency.pth'
if not os.path.exists(pretrained_path):
    raise FileNotFoundError(f'事前学習済み重みが見つかりません: {pretrained_path}')

model3D = vxm.networks.VxmDense_128_256_256((128, 256, 256), nb_features, int_steps=0).to(device)
model3D.load_state_dict(torch.load(pretrained_path, map_location=device, weights_only=True))
optimizer = optim.Adam(model3D.parameters(), lr=1e-5)
transformer = vxm.layers.SpatialTransformer((64, 128, 128)).to(device)

epochs = 30000
losses = []
loss_images_ab = []
loss_images_ba = []
loss_inverses = []

for epoch in tqdm(range(epochs), desc='DWT + inverse-consistency fine-tuning'):
    train_batch, _ = next(train_generator)
    moving_images = torch.as_tensor(train_batch[0], dtype=torch.float32, device=device)
    fixed_images = torch.as_tensor(train_batch[1], dtype=torch.float32, device=device)

    moving_w = down_sampling_3d(analysis_filter_3d(moving_images, analysis)).to(device)
    fixed_w = down_sampling_3d(analysis_filter_3d(fixed_images, analysis)).to(device)
    optimizer.zero_grad()

    flow_ab = model3D(moving_w, fixed_w)
    moved_ab, _ = synthesis_filter_3d(up_sampling_3d(transformer(moving_w, flow_ab)), synthesis_filters)
    flow_ba = model3D(fixed_w, moving_w)
    moved_ba, _ = synthesis_filter_3d(up_sampling_3d(transformer(fixed_w, flow_ba)), synthesis_filters)

    loss_image_ab = MSE_Loss(fixed_images, moved_ab)
    loss_image_ba = MSE_Loss(moving_images, moved_ba)
    loss_inverse = inverse_consistency_loss(flow_ab, flow_ba)
    loss = loss_image_ab + loss_image_ba + INVERSE_CONSISTENCY_WEIGHT * loss_inverse
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    loss_images_ab.append(loss_image_ab.item())
    loss_images_ba.append(loss_image_ba.item())
    loss_inverses.append(loss_inverse.item())

    if (epoch + 1) % 100 == 0:
        torch.save(model3D.state_dict(), 'model_analysis_pipeline_finetuned_inverse_consistency.pth')
        print(
            f'Fine-tune {epoch + 1}/{epochs}, total={loss:.5f}, '
            f'A→B={loss_image_ab:.5f}, B→A={loss_image_ba:.5f}, inverse={loss_inverse:.5f}'
        )
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        for axis, image, title in zip(axes, [moving_images, fixed_images, moved_ab], ['Moving', 'Fixed', 'Moved (A→B)']):
            axis.imshow(image[0, 0, image.shape[2] // 2].detach().cpu(), cmap='gray')
            axis.set_title(title)
            axis.axis('off')
        plt.show()

torch.save(model3D.state_dict(), 'model_analysis_pipeline_finetuned_inverse_consistency_final.pth')
print('逆写像整合性つきファインチューニング済みモデルを保存しました。')


In [14]:
import os

for drive in ["D:\\", "C:\\"]:
    for root, dirs, files in os.walk(drive):
        for file in files:
            if "wavelet" in file.lower() and file.endswith(".pth"):
                print(os.path.join(root, file))

D:\Saito\model_wavelet_128_256_256.pth
D:\Saito\model_wavelet_finetune_final.pth
D:\Saito\model_wavelet_pretrain_checkpoint.pth
D:\Saito\model_wavelet_pretrain_final.pth
D:\Saito\model_wavelet_pretrain_final_80000.pth
D:\Saito\model_wavelet_pretrain_restart_checkpoint.pth
D:\Saito\Saito_model_wavelet_128_256_256.pth
D:\Yamato\model_VXM_3D_MInoBed_WaveletEncorder.pth
D:\Yamato\model_VXM_3D_MInoBed_WaveletTest.pth
D:\Yamato\model_VXM_3D_weights_Wavelet.pth
C:\Users\user\OneDrive\ドキュメント\model_VXM_3D_MInoBed_WaveletTest.pth


In [15]:
import torch, os

save_path = r"D:\Saito\model_wavelet_128_256_256.pth"
torch.save(model3D.state_dict(), save_path)

print(os.path.exists(save_path))
print(save_path)

True
D:\Saito\model_wavelet_128_256_256.pth


In [ ]:
# Check_Perfect_Reconstruction.py
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

pr_names = ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

class CheckPRHaar3DAnalysis(nn.Module):
    def __init__(self):
        super().__init__()
        hL = torch.tensor([1.0, 1.0], dtype=torch.float32) / math.sqrt(2)
        hH = torch.tensor([1.0, -1.0], dtype=torch.float32) / math.sqrt(2)

        filters = []
        filter_names = []
        for z_name, z_filter in zip(['L', 'H'], [hL, hH]):
            for y_name, y_filter in zip(['L', 'H'], [hL, hH]):
                for x_name, x_filter in zip(['L', 'H'], [hL, hH]):
                    kernel = (
                        z_filter[:, None, None]
                        * y_filter[None, :, None]
                        * x_filter[None, None, :]
                    )
                    filters.append(kernel)
                    filter_names.append(z_name + y_name + x_name)

        weight = torch.stack(filters, dim=0).unsqueeze(1)
        self.register_buffer('weight', weight)
        self.names = filter_names

    def forward(self, x):
        x_pad = F.pad(x, (0, 1, 0, 1, 0, 1))
        w = F.conv3d(x_pad, self.weight, stride=1, padding=0)
        return w

def check_pr_downsample_3d(w):
    return w[:, :, ::2, ::2, ::2]

def check_pr_upsample_3d(w_down):
    B, C, D, H, W = w_down.shape
    w_up = torch.zeros(B, C, D * 2, H * 2, W * 2, dtype=w_down.dtype, device=w_down.device)
    w_up[:, :, ::2, ::2, ::2] = w_down
    return w_up

def check_pr_make_3d_filter(fz, fy, fx):
    return fz[:, None, None] * fy[None, :, None] * fx[None, None, :]

class CheckPRHaar3DSynthesis(nn.Module):
    def __init__(self):
        super().__init__()
        sqrt2 = math.sqrt(2.0)
        low = torch.tensor([1.0, 1.0], dtype=torch.float32) / sqrt2
        high = torch.tensor([1.0, -1.0], dtype=torch.float32) / sqrt2

        filters = torch.stack([
            check_pr_make_3d_filter(low, low, low),
            check_pr_make_3d_filter(low, low, high),
            check_pr_make_3d_filter(low, high, low),
            check_pr_make_3d_filter(low, high, high),
            check_pr_make_3d_filter(high, low, low),
            check_pr_make_3d_filter(high, low, high),
            check_pr_make_3d_filter(high, high, low),
            check_pr_make_3d_filter(high, high, high),
        ], dim=0)

        filters = torch.flip(filters, dims=[1, 2, 3]).unsqueeze(1)
        self.register_buffer('filters', filters)

    def forward(self, w_up):
        B, C, D, H, W = w_up.shape
        filtered_bands = []
        for i, name in enumerate(pr_names):
            band = w_up[:, i:i + 1, :, :, :]
            kernel = self.filters[i:i + 1]
            filtered = F.conv3d(band, kernel, stride=1, padding=1)
            filtered = filtered[:, :, :D, :H, :W]
            filtered_bands.append(filtered)
            print(name, 'Synthesis後:', filtered.shape)

        filtered_bands = torch.cat(filtered_bands, dim=1)
        reconstructed = torch.sum(filtered_bands, dim=1, keepdim=True)
        return filtered_bands, reconstructed

def check_pr_frequency_response(h, omega):
    n = np.arange(len(h))
    response = np.sum(h[None, :] * np.exp(-1j * omega[:, None] * n[None, :]), axis=1)
    return response

In [35]:
# Check_Perfect_Reconstruction.py: reconstruction check
check_pr_x = torch.as_tensor(
    x_train[0:1],
    dtype=torch.float32,
    device=device
).unsqueeze(1)

print('\n===================================')
print('入力')
print('===================================')
print('元画像:', check_pr_x.shape)

print('\n===================================')
print('1. Analysis Filter')
print('===================================')
check_pr_analysis = CheckPRHaar3DAnalysis().to(device)
check_pr_w = check_pr_analysis(check_pr_x)
print('Analysis後:', check_pr_w.shape)
print('周波数成分:', check_pr_analysis.names)

print('\n===================================')
print('2. Downsampling')
print('===================================')
check_pr_w_down = check_pr_downsample_3d(check_pr_w)
print('Downsampling後:', check_pr_w_down.shape)

print('\n===================================')
print('3. Upsampling')
print('===================================')
check_pr_w_up = check_pr_upsample_3d(check_pr_w_down)
print('Upsampling後:', check_pr_w_up.shape)

check_pr_up_error = torch.abs(check_pr_w_up[:, :, ::2, ::2, ::2] - check_pr_w_down)
print('Upsampling配置確認 平均誤差:', check_pr_up_error.mean().item())
print('Upsampling配置確認 最大誤差:', check_pr_up_error.max().item())

print('\n===================================')
print('4. Synthesis Filter')
print('===================================')
check_pr_synthesis = CheckPRHaar3DSynthesis().to(device)
check_pr_filtered_bands, check_pr_reconstructed = check_pr_synthesis(check_pr_w_up)
print('\nSynthesis後8成分:', check_pr_filtered_bands.shape)
print('再構成画像:', check_pr_reconstructed.shape)

print('\n===================================')
print('5. Reconstruction Error')
print('===================================')

if check_pr_x.shape != check_pr_reconstructed.shape:
    print('サイズが一致していません')
    print('Original:', check_pr_x.shape)
    print('Reconstructed:', check_pr_reconstructed.shape)
else:
    print('サイズ一致')
    check_pr_diff = check_pr_x - check_pr_reconstructed
    check_pr_absolute_error = torch.abs(check_pr_diff)
    check_pr_mae = torch.mean(check_pr_absolute_error)
    check_pr_max_error = torch.max(check_pr_absolute_error)
    check_pr_mse = torch.mean(check_pr_diff ** 2)
    check_pr_relative_error = torch.norm(check_pr_diff) / torch.norm(check_pr_x)
    print('\n===== 再構成誤差 =====')
    print('MAE:', check_pr_mae.item())
    print('最大絶対誤差:', check_pr_max_error.item())
    print('MSE:', check_pr_mse.item())
    print('相対誤差:', check_pr_relative_error.item())

    if check_pr_max_error.item() > 0:
         check_pr_error_order = int(np.floor(np.log10(check_pr_max_error.item())))
         check_pr_error_scale = 10.0 ** (-check_pr_error_order)
    else:
         check_pr_error_order = 0
         check_pr_error_scale = 1.0

    print('誤差表示スケール:', f'x 1e{-check_pr_error_order}' if check_pr_max_error.item() > 0 else 'no scaling')

print('\n===================================')
print('6. Save Results')
print('===================================')
np.save(r'D:\Saito\wavelet_reconstructed.npy', check_pr_reconstructed.detach().cpu().numpy())
print('再構成画像を保存しました')

if check_pr_x.shape == check_pr_reconstructed.shape:
    check_pr_slice_index = check_pr_x.shape[2] // 2

    plt.figure(figsize=(8, 8))
    plt.imshow(check_pr_x[0, 0, check_pr_slice_index].detach().cpu().numpy(), cmap='gray')
    plt.title('Original Image')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_original.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(check_pr_reconstructed[0, 0, check_pr_slice_index].detach().cpu().numpy(), cmap='gray')
    plt.title('Reconstructed Image')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_reconstructed.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(
         check_pr_absolute_error[0, 0, check_pr_slice_index].detach().cpu().numpy(),
         cmap='inferno',
         vmin=0.0,
         vmax=check_pr_max_error.item()
     )
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title('Absolute Reconstruction Error')
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_error.png', dpi=300, bbox_inches='tight')
    plt.close()

    plt.figure(figsize=(8, 8))
    plt.imshow(
       (check_pr_absolute_error[0, 0, check_pr_slice_index] * check_pr_error_scale).detach().cpu().numpy(),
       cmap='inferno'
     )
    plt.colorbar(fraction=0.046, pad=0.04)
    plt.title(
         f'Scaled Error (x 1e{-check_pr_error_order})'
         if check_pr_max_error.item() > 0
         else 'Scaled Error'
     )
    plt.axis('off')
    plt.savefig(r'D:\Saito\wavelet_error_scaled.png', dpi=300, bbox_inches='tight')
    plt.close()


入力
元画像: torch.Size([1, 1, 128, 256, 256])

1. Analysis Filter
Analysis後: torch.Size([1, 8, 128, 256, 256])
周波数成分: ['LLL', 'LLH', 'LHL', 'LHH', 'HLL', 'HLH', 'HHL', 'HHH']

2. Downsampling
Downsampling後: torch.Size([1, 8, 64, 128, 128])

3. Upsampling
Upsampling後: torch.Size([1, 8, 128, 256, 256])
Upsampling配置確認 平均誤差: 0.0
Upsampling配置確認 最大誤差: 0.0

4. Synthesis Filter
LLL Synthesis後: torch.Size([1, 1, 128, 256, 256])
LLH Synthesis後: torch.Size([1, 1, 128, 256, 256])
LHL Synthesis後: torch.Size([1, 1, 128, 256, 256])
LHH Synthesis後: torch.Size([1, 1, 128, 256, 256])
HLL Synthesis後: torch.Size([1, 1, 128, 256, 256])
HLH Synthesis後: torch.Size([1, 1, 128, 256, 256])
HHL Synthesis後: torch.Size([1, 1, 128, 256, 256])
HHH Synthesis後: torch.Size([1, 1, 128, 256, 256])

Synthesis後8成分: torch.Size([1, 8, 128, 256, 256])
再構成画像: torch.Size([1, 1, 128, 256, 256])

5. Reconstruction Error
サイズ一致

===== 再構成誤差 =====
MAE: 3.967887707290174e-08
最大絶対誤差: 4.172325134277344e-07
MSE: 5.188463472473011e-15
相対誤差:

In [36]:
if check_pr_x.shape == check_pr_reconstructed.shape:
    check_pr_slice_index = check_pr_x.shape[2] // 2

    # ========================================
    # 元画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_x[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='gray'
    )

    plt.title('Original Image')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_original.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 再構成画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_reconstructed[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='gray'
    )

    plt.title('Reconstructed Image')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_reconstructed.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 絶対誤差画像
    # ========================================
    plt.figure(figsize=(8, 8))

    plt.imshow(
        check_pr_absolute_error[
            0,
            0,
            check_pr_slice_index
        ].detach().cpu().numpy(),
        cmap='inferno',
        vmin=0.0,
        vmax=check_pr_max_error.item()
    )

    plt.colorbar(
        fraction=0.046,
        pad=0.04
    )

    plt.title('Absolute Reconstruction Error')
    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_error.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()


    # ========================================
    # 拡大表示した誤差画像
    # ========================================
    plt.figure(figsize=(8, 8))

    check_pr_scaled_error = (
        check_pr_absolute_error[
            0,
            0,
            check_pr_slice_index
        ]
        * check_pr_error_scale
    ).detach().cpu().numpy()

    plt.imshow(
        check_pr_scaled_error,
        cmap='inferno'
    )

    plt.colorbar(
        fraction=0.046,
        pad=0.04
    )

    if check_pr_max_error.item() > 0:
        plt.title(
            f'Scaled Error (x 1e{-check_pr_error_order})'
        )
    else:
        plt.title('Scaled Error')

    plt.axis('off')

    plt.savefig(
        r'D:\Saito\wavelet_error_scaled.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.close()

    print('画像を保存しました')

画像を保存しました


In [37]:
# Check_Perfect_Reconstruction.py: perfect reconstruction conditions
print('\n===================================')
print('7. Perfect Reconstruction Conditions')
print('===================================')

check_pr_sqrt2 = np.sqrt(2.0)
check_pr_h0 = np.array([1.0, 1.0]) / check_pr_sqrt2
check_pr_h1 = np.array([-1.0, 1.0]) / check_pr_sqrt2
check_pr_f0 = np.array([1.0, 1.0]) / check_pr_sqrt2
check_pr_f1 = np.array([1.0, -1.0]) / check_pr_sqrt2

check_pr_N = 2048
check_pr_omega = np.linspace(-np.pi, np.pi, check_pr_N, endpoint=False)

check_pr_H0 = check_pr_frequency_response(check_pr_h0, check_pr_omega)
check_pr_H1 = check_pr_frequency_response(check_pr_h1, check_pr_omega)
check_pr_F0 = check_pr_frequency_response(check_pr_f0, check_pr_omega)
check_pr_F1 = check_pr_frequency_response(check_pr_f1, check_pr_omega)
check_pr_H0_minus = check_pr_frequency_response(check_pr_h0, check_pr_omega + np.pi)
check_pr_H1_minus = check_pr_frequency_response(check_pr_h1, check_pr_omega + np.pi)

check_pr_alias_term = check_pr_H0_minus * check_pr_F0 + check_pr_H1_minus * check_pr_F1
check_pr_alias_max = np.max(np.abs(check_pr_alias_term))
print('\n-----------------------------------')
print('条件1: Alias Cancellation')
print('-----------------------------------')
print('最大Alias成分:', check_pr_alias_max)

check_pr_T = check_pr_H0 * check_pr_F0 + check_pr_H1 * check_pr_F1
check_pr_magnitude = np.abs(check_pr_T)
check_pr_magnitude_min = np.min(check_pr_magnitude)
check_pr_magnitude_max = np.max(check_pr_magnitude)
check_pr_magnitude_variation = check_pr_magnitude_max - check_pr_magnitude_min
print('\n-----------------------------------')
print('条件2: Amplitude Distortion')
print('-----------------------------------')
print('Magnitude min:', check_pr_magnitude_min)
print('Magnitude max:', check_pr_magnitude_max)
print('Magnitude variation:', check_pr_magnitude_variation)

check_pr_phase = np.unwrap(np.angle(check_pr_T))
check_pr_phase_coef = np.polyfit(check_pr_omega, check_pr_phase, 1)
check_pr_phase_fit = np.polyval(check_pr_phase_coef, check_pr_omega)
check_pr_phase_error = check_pr_phase - check_pr_phase_fit
check_pr_max_phase_error = np.max(np.abs(check_pr_phase_error))
print('\n-----------------------------------')
print('条件3: Phase Distortion')
print('-----------------------------------')
print('Phase slope:', check_pr_phase_coef[0])
print('最大直線位相誤差:', check_pr_max_phase_error)

check_pr_tolerance = 1e-10
print('\n===================================')
print('PR Condition Results')
print('===================================')
print('条件1 Alias Cancellation:', 'OK' if check_pr_alias_max < check_pr_tolerance else 'NG')
print('条件2 Amplitude Distortion:', 'OK' if check_pr_magnitude_variation < check_pr_tolerance else 'NG')
print('条件3 Phase Distortion:', 'OK' if check_pr_max_phase_error < check_pr_tolerance else 'NG')

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, np.abs(check_pr_alias_term))
plt.xlabel('Angular Frequency ω')
plt.ylabel('|Alias Term|')
plt.title('Alias Cancellation')
plt.grid()
plt.savefig(r'D:\Saito\pr_alias.png', dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, check_pr_magnitude)
plt.xlabel('Angular Frequency ω')
plt.ylabel('|T(e^jω)|')
plt.title('Distortion Transfer Function Magnitude')
plt.grid()
plt.savefig(r'D:\Saito\pr_amplitude.png', dpi=300, bbox_inches='tight')
plt.close()

plt.figure(figsize=(8, 5))
plt.plot(check_pr_omega, check_pr_phase, label='Actual Phase')
plt.plot(check_pr_omega, check_pr_phase_fit, linestyle='--', label='Linear Fit')
plt.xlabel('Angular Frequency ω')
plt.ylabel('Phase [rad]')
plt.title('Phase Response')
plt.legend()
plt.grid()
plt.savefig(r'D:\Saito\pr_phase.png', dpi=300, bbox_inches='tight')
plt.close()

print('\nPR条件確認用グラフを保存しました')
print('\n処理完了')


7. Perfect Reconstruction Conditions

-----------------------------------
条件1: Alias Cancellation
-----------------------------------
最大Alias成分: 3.554447978966673e-16

-----------------------------------
条件2: Amplitude Distortion
-----------------------------------
Magnitude min: 1.999999999999999
Magnitude max: 2.0000000000000004
Magnitude variation: 1.5543122344752192e-15

-----------------------------------
条件3: Phase Distortion
-----------------------------------
Phase slope: -1.0
最大直線位相誤差: 8.881784197001252e-16

PR Condition Results
条件1 Alias Cancellation: OK
条件2 Amplitude Distortion: OK
条件3 Phase Distortion: OK

PR条件確認用グラフを保存しました

処理完了


In [ ]:
# ファインチューニングは上の3万epochセルに統合済みです。ここでは何もしません。

In [ ]:
# チェックポイント保存は上の3万epochセルで100epochごとに実行されます。

In [ ]:
# 最終モデル保存は上の3万epochセルの完了時に実行されます。